In [1]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "12345678"
auth=(username, password)
driver = GraphDatabase.driver(uri, auth=(username, password))

In [2]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Neo4jMapReduce") \
    .master("local[*]") \
    .getOrCreate()

ImportError: cannot import name 'PySparkRuntimeError' from 'pyspark.errors' (unknown location)

In [ ]:
def Map(driver):

    query = """
    MATCH (c)
    WHERE c.kind = "Compound"
    OPTIONAL MATCH (c)-[r]->()
    RETURN c.name AS Compound, r.metaedge AS metaedge
    """
    with driver.session() as session:
        result = [dict(record) for record in session.run(query)]

    print("Result sample:", result[:5])
    rdd = spark.sparkContext.parallelize(result)
    pairs = rdd.map(lambda x: (x['Compound'], x['metaedge']))
    return pairs

In [3]:

query = """
MATCH (c)
WHERE c.kind = "Compound"
OPTIONAL MATCH (c)-[r]->()
RETURN c.name AS Compound, r.metaedge AS metaedge
"""
with driver.session() as session:
    result = [dict(record) for record in session.run(query)]

print("Result sample:", result[:5])
# First, create the RDD from your results
rdd = spark.sparkContext.parallelize(result)

# Try a simpler operation first to test the RDD
simple_mapped = rdd.map(lambda x: (x['Compound'], x['metaedge']))

# Collect locally first instead of using take() which might be triggering the error
try:
    collected_data = simple_mapped.collect()
    print("First 5 pairs:", collected_data[:5])
except Exception as e:
    print("Collection error:", str(e))

Result sample: [{'Compound': 'Goserelin', 'metaedge': 'CrC'}, {'Compound': 'Goserelin', 'metaedge': 'CrC'}, {'Compound': 'Goserelin', 'metaedge': 'CrC'}, {'Compound': 'Goserelin', 'metaedge': 'CrC'}, {'Compound': 'Goserelin', 'metaedge': 'CrC'}]
Collection error: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 0.0 failed 1 times, most recent failure: Lost task 3.0 in stage 0.0 (TID 3) (DESKTOP-E6MA3J1 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner

In [61]:
def Sort(pairs):

    gene_types = ['CbG', 'CuG', 'CdG']
    genes = pairs.filter(lambda x: x[1] in gene_types).map(lambda x: (x[0], 1))

    disease_types = ['CtD', 'CpD']
    diseases = pairs.filter(lambda x: x[1] in disease_types).map(lambda x: (x[0], 1))
    
    return genes, diseases


In [66]:
def Reduce(genes, diseases):

    gene_counts = genes.reduceByKey(lambda a, b: a + b)
    return gene_counts
    

   
   


In [67]:
def MapReduce(driver):
    pairs = Map(driver)
    genes, diseases = Sort(pairs)
    reduced_pairs = Reduce(genes, diseases)
    return reduced_pairs

In [68]:
result = MapReduce(driver)
print(result.take(5))

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 14.0 failed 1 times, most recent failure: Lost task 1.0 in stage 14.0 (TID 51) (DESKTOP-E6MA3J1 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "C:\Users\Jason\anaconda3\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1100, in main
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 10) than that in driver 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apach

In [ ]:
spark.stop()
driver.close()